# Assignment: Multi-Asset Portfolios — Correlation, Clustering & Dimension Reduction
**Navigating High-Dimensional Markets**

| Part | Focus | Methods |
|------|-------|---------|
| 1 | Correlation Matrix & Diversification | Heatmap, rolling avg correlation |
| 2 | Tail Dependency & Partial Correlation | Tail dependence coefficients, partial corr |
| 3 | Clustering Techniques | HCA dendrogram, K-Means, elbow, curse of dimensionality |
| 4 | Dimension Reduction & MST | PCA, RMT cleaning, Minimum Spanning Tree |

In [27]:
import numpy as np
import pandas as pd
from scipy import stats, cluster, spatial
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA as SKPCA
import plotly.graph_objects as go
import plotly.subplots as ps
import warnings
warnings.filterwarnings('ignore')

SEED   = 42
rng    = np.random.default_rng(SEED)
ANNUAL = 252

# ── 20 assets across 4 classes (5 each) ──────────────────────────────────────
ASSET_CLASSES = {
    'Equity'    : ['EQ01','EQ02','EQ03','EQ04','EQ05'],
    'Bond'      : ['BD01','BD02','BD03','BD04','BD05'],
    'Commodity' : ['CM01','CM02','CM03','CM04','CM05'],
    'FX'        : ['FX01','FX02','FX03','FX04','FX05'],
}
ASSETS = [a for v in ASSET_CLASSES.values() for a in v]
N      = len(ASSETS)   # 20
T      = 756           # ~3 years of daily bars
CLASS_COLORS = {
    'Equity':'#00d4aa', 'Bond':'lightsalmon',
    'Commodity':'cornflowerblue', 'FX':'#ff6b6b'
}

print(f'Universe  : {N} assets | {T} days')
print('Classes   :', {k: len(v) for k,v in ASSET_CLASSES.items()})

Universe  : 20 assets | 756 days
Classes   : {'Equity': 5, 'Bond': 5, 'Commodity': 5, 'FX': 5}


---
## Part 1 | Correlation Matrix & Diversification

We simulate daily returns with **four latent risk factors** (one per asset class) plus a common market factor. This produces a realistic block-diagonal correlation structure.

$$\sigma_p^2 = w^\top \Sigma\, w = \sum_i w_i^2\sigma_i^2 + 2\sum_{i<j}w_iw_j\sigma_i\sigma_j\rho_{ij}$$

Diversification benefit is maximised when off-diagonal correlations are low.  
During stress events correlations converge toward 1, eliminating the free lunch.

In [28]:
# ── Four class factors + market factor + idiosyncratic noise ─────────────────
FACTOR_VOLS = {'Equity':0.015, 'Bond':0.006, 'Commodity':0.018, 'FX':0.008}
MARKET_VOL  = 0.008
IDIO_VOL    = 0.010

market_factor = rng.normal(0, MARKET_VOL, T)
class_factors = {cls: rng.normal(0, vol, T) for cls, vol in FACTOR_VOLS.items()}

ret_dict = {}
for cls, names in ASSET_CLASSES.items():
    for a in names:
        beta_mkt = rng.uniform(0.3, 0.8)
        beta_cls = rng.uniform(0.6, 1.0)
        idio     = rng.normal(0, IDIO_VOL, T)
        ret_dict[a] = beta_mkt*market_factor + beta_cls*class_factors[cls] + idio

returns = pd.DataFrame(ret_dict)
corr    = returns.corr()
asset_class = {a: cls for cls, names in ASSET_CLASSES.items() for a in names}

avg_within = np.mean([corr.loc[a,b] for cls,names in ASSET_CLASSES.items()
                       for a in names for b in names if a!=b])
avg_across = np.mean([corr.loc[a,b] for cls1,n1 in ASSET_CLASSES.items()
                       for cls2,n2 in ASSET_CLASSES.items() if cls1!=cls2
                       for a in n1 for b in n2])
print('Average within-class correlation : {:.3f}'.format(avg_within))
print('Average across-class correlation : {:.3f}'.format(avg_across))
print('Diversification ratio            : {:.2f}x'.format(avg_within / max(avg_across,1e-6)))

Average within-class correlation : 0.498
Average across-class correlation : 0.095
Diversification ratio            : 5.27x


In [29]:
fig = go.Figure(go.Heatmap(
    z=corr.values, x=ASSETS, y=ASSETS,
    colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
    colorbar=dict(title='Pearson rho')
))
class_sizes = [len(v) for v in ASSET_CLASSES.values()]
for b in np.cumsum(class_sizes[:-1]) - 0.5:
    for shape in [
        dict(type='line', x0=b, x1=b, y0=-0.5, y1=N-0.5),
        dict(type='line', x0=-0.5, x1=N-0.5, y0=b, y1=b),
    ]:
        fig.add_shape(**shape, line=dict(color='white', width=2))
for i, (cls, col) in enumerate(CLASS_COLORS.items()):
    mid = sum(class_sizes[:i]) + class_sizes[i]/2 - 0.5
    fig.add_annotation(x=mid, y=N+0.5, text=cls, showarrow=False,
                        font=dict(color=col, size=11))
fig.update_layout(
    title='Correlation Matrix — Block-Diagonal Structure by Asset Class',
    template='plotly_dark', width=620, height=580
)
fig.show()

In [30]:
# ── Rolling 63-day average pairwise correlation ───────────────────────────────
ROLL         = 63
STRESS_START = 350
STRESS_END   = 420

# Inject common stress shock
ret_stress = returns.copy()
shock = rng.normal(-0.002, 0.025, (STRESS_END - STRESS_START, 1))
ret_stress.iloc[STRESS_START:STRESS_END] += shock

def rolling_avg_corr(df, roll):
    out = []
    vals = df.values
    for t in range(roll, len(df)):
        w = vals[t-roll:t]
        c = np.corrcoef(w.T)
        out.append(c[np.triu_indices(N, k=1)].mean())
    return pd.Series(out, index=range(roll, len(df)))

rc_normal = rolling_avg_corr(returns,    ROLL)
rc_stress = rolling_avg_corr(ret_stress, ROLL)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rc_normal.index, y=rc_normal.values,
                          name='Normal', line=dict(color='#00d4aa', width=1.5)))
fig.add_trace(go.Scatter(x=rc_stress.index, y=rc_stress.values,
                          name='With stress episode', line=dict(color='#ff6b6b', width=1.5)))
fig.add_vrect(x0=STRESS_START, x1=STRESS_END,
               fillcolor='rgba(255,107,107,0.10)', line_width=0,
               annotation_text='Stress', annotation_position='top left')
fig.add_hline(y=rc_normal.mean(), line_dash='dash', line_color='#00d4aa', opacity=0.5,
               annotation_text='Avg normal', annotation_position='bottom right')
fig.update_layout(
    title='Rolling 63-day Average Pairwise Correlation — Diversification Collapses in Stress',
    xaxis_title='Day', yaxis_title='Average Correlation',
    template='plotly_dark'
)
fig.show()

---
## Part 2 | Tail Dependency & Partial Correlation

**Tail Dependence Coefficient (TDC)**: empirical probability that both assets fall into the same extreme quantile simultaneously, normalised by the quantile probability.

$$\lambda_L = P\!\left(X < F_X^{-1}(q),\; Y < F_Y^{-1}(q)\right) / q \quad q \to 0$$

A value > 1 means co-movement is more frequent than if returns were independent.  
**Asymmetry**: crashes (lower tail) typically show stronger co-movement than rallies.

**Partial correlation** removes the shared market-beta effect, isolating *direct* pairwise linkages.

In [31]:
Q = 0.10

def tdc(r1, r2, q, lower=True):
    if lower:
        mask = (r1 <= np.quantile(r1, q)) & (r2 <= np.quantile(r2, q))
    else:
        mask = (r1 >= np.quantile(r1, 1-q)) & (r2 >= np.quantile(r2, 1-q))
    return mask.sum() / (q * len(r1))

rv = returns.values.T
wl, wu, al, au = [], [], [], []
for i in range(N):
    for j in range(i+1, N):
        ci, cj = asset_class[ASSETS[i]], asset_class[ASSETS[j]]
        ld = tdc(rv[i], rv[j], Q, lower=True)
        ud = tdc(rv[i], rv[j], Q, lower=False)
        if ci == cj: wl.append(ld); wu.append(ud)
        else:        al.append(ld); au.append(ud)

labels = ['Within-class\nLower tail','Within-class\nUpper tail',
           'Across-class\nLower tail','Across-class\nUpper tail']
vals   = [np.mean(wl), np.mean(wu), np.mean(al), np.mean(au)]
cols   = ['#ff6b6b','#00d4aa','#ff6b6b','#00d4aa']

fig = go.Figure(go.Bar(
    x=labels, y=vals, marker_color=cols,
    text=['{:.2f}'.format(v) for v in vals], textposition='outside'
))
fig.add_hline(y=1.0, line_dash='dash', line_color='white', opacity=0.4,
               annotation_text='Independent baseline (=1)', annotation_position='top right')
fig.update_layout(
    title='Tail Dependence Coefficients (q={}%) -- Lower vs Upper Tail'.format(int(Q*100)),
    yaxis_title='TDC  (>1 = excess co-movement)',
    template='plotly_dark'
)
fig.show()
print('Within-class asymmetry: lower={:.2f}  upper={:.2f}'.format(np.mean(wl), np.mean(wu)))

Within-class asymmetry: lower=0.36  upper=0.34


In [32]:
# ── Partial correlation controlling for equal-weight market proxy ─────────────
mkt = returns.mean(axis=1).values

def partial_corr(ri, rj, ctrl):
    res_i = ri - np.polyval(np.polyfit(ctrl, ri, 1), ctrl)
    res_j = rj - np.polyval(np.polyfit(ctrl, rj, 1), ctrl)
    return np.corrcoef(res_i, res_j)[0, 1]

pcorr = np.eye(N)
for i in range(N):
    for j in range(i+1, N):
        pc = partial_corr(rv[i], rv[j], mkt)
        pcorr[i,j] = pcorr[j,i] = pc

fig = ps.make_subplots(rows=1, cols=2,
                        subplot_titles=['Full Pearson Correlation',
                                        'Partial Correlation (market removed)'])
for col, mat in [(1, corr.values), (2, pcorr)]:
    fig.add_trace(
        go.Heatmap(z=mat, x=ASSETS, y=ASSETS, colorscale='RdBu_r',
                    zmid=0, zmin=-1, zmax=1, showscale=(col==2)),
        row=1, col=col)
fig.update_layout(
    title='Full vs. Partial Correlation -- Removing Spurious Market-Beta Linkages',
    template='plotly_dark', height=500
)
fig.show()
print('Mean |Pearson|  : {:.3f}'.format(np.abs(corr.values[np.triu_indices(N,1)]).mean()))
print('Mean |Partial|  : {:.3f}'.format(np.abs(pcorr[np.triu_indices(N,1)]).mean()))

Mean |Pearson|  : 0.179
Mean |Partial|  : 0.210


---
## Part 3 | Clustering & The Curse of Dimensionality

Clustering uses the **correlation distance** $d_{ij} = \sqrt{2(1-\rho_{ij})}$ to group assets by risk DNA:

- **HCA** (Ward linkage): bottom-up agglomerative, no need to pre-specify K, visualised as a dendrogram.
- **K-Means**: minimise WCSS $= \sum_{k=1}^K \sum_{x \in S_k} \|x - \mu_k\|^2$; elbow method selects K.

**Curse of dimensionality**: covariance has $N(N+1)/2$ parameters. When $N \approx T$ the sample matrix becomes ill-conditioned and MV weights explode.

In [33]:
dist_mat  = np.sqrt(2 * (1 - corr.values))
condensed = spatial.distance.squareform(dist_mat, checks=False)
linkage   = cluster.hierarchy.linkage(condensed, method='ward')
dend      = cluster.hierarchy.dendrogram(linkage, labels=ASSETS, no_plot=True)

icoord = np.array(dend['icoord'])
dcoord = np.array(dend['dcoord'])
leaves = dend['ivl']

fig = go.Figure()
for xs, ys in zip(icoord, dcoord):
    fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines',
                              line=dict(color='#00d4aa', width=1.2),
                              showlegend=False))

# Colour leaf labels by asset class
all_xs = sorted(set(icoord[:,0].tolist() + icoord[:,2].tolist()))
for x, lbl in zip(all_xs[:N], leaves):
    cls = asset_class[lbl]
    fig.add_annotation(x=x, y=-0.03, text=lbl, textangle=-90, showarrow=False,
                        yref='paper', font=dict(color=CLASS_COLORS[cls], size=10))

fig.update_layout(
    title='Hierarchical Clustering Dendrogram (Ward linkage, correlation distance)<br>'
          '<sup>Colour = asset class — correctly recovered without labels</sup>',
    xaxis=dict(showticklabels=False), yaxis_title='Ward Distance',
    template='plotly_dark', height=460
)
fig.show()

In [34]:
# ── K-Means on PCA-compressed features (first 5 PCs) ─────────────────────────
pca_feat = SKPCA(n_components=5, random_state=SEED).fit_transform(returns.T)

K_RANGE  = range(2, 9)
inertias = [KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(pca_feat).inertia_
             for k in K_RANGE]

km4      = KMeans(n_clusters=4, random_state=SEED, n_init=10).fit(pca_feat)
km_lbls  = km4.labels_

fig = ps.make_subplots(rows=1, cols=2,
                        subplot_titles=['Elbow Curve (WCSS vs K)', 'K-Means K=4 (PC1 vs PC2)'])
fig.add_trace(go.Scatter(x=list(K_RANGE), y=inertias, mode='lines+markers',
                          line=dict(color='lightsalmon', width=2),
                          marker=dict(size=6), showlegend=False), row=1, col=1)
fig.add_vline(x=4, line_dash='dash', line_color='#00d4aa',
               annotation_text='K=4', annotation_position='top right', row=1, col=1)

KM_COLS = ['#00d4aa','lightsalmon','cornflowerblue','#ff6b6b']
for k in range(4):
    mask = km_lbls == k
    fig.add_trace(go.Scatter(
        x=pca_feat[mask,0], y=pca_feat[mask,1],
        mode='markers+text',
        marker=dict(color=KM_COLS[k], size=9),
        text=np.array(ASSETS)[mask].tolist(),
        textposition='top center', textfont=dict(size=9),
        name='Cluster {}'.format(k+1)
    ), row=1, col=2)

fig.update_xaxes(title_text='K', row=1, col=1)
fig.update_yaxes(title_text='WCSS (inertia)', row=1, col=1)
fig.update_xaxes(title_text='PC1', row=1, col=2)
fig.update_yaxes(title_text='PC2', row=1, col=2)
fig.update_layout(template='plotly_dark', height=430,
                   title='K-Means Clustering -- Elbow Method and 2D Projection')
fig.show()

# Alignment check
true_idx = [list(ASSET_CLASSES.keys()).index(asset_class[a]) for a in ASSETS]
ct = pd.crosstab(pd.Series(true_idx, name='True class'),
                  pd.Series(km_lbls,  name='K-Means cluster'))
print('Contingency table (true class vs K-Means):')
print(ct)

Contingency table (true class vs K-Means):
K-Means cluster  0  1  2  3
True class                 
0                0  0  5  0
1                5  0  0  0
2                0  5  0  0
3                0  0  0  5


In [35]:
# ── Condition number of sample covariance as N grows (T fixed) ───────────────
T_fixed  = 252
N_range  = np.arange(5, 246, 5)
cond_num = []
for n in N_range:
    r = rng.normal(0, 0.01, (T_fixed, n))
    ev = np.linalg.eigvalsh(np.cov(r.T))
    cond_num.append(ev[-1] / max(ev[0], 1e-12))

fig = go.Figure(go.Scatter(
    x=N_range, y=cond_num, mode='lines',
    line=dict(color='#ff6b6b', width=2)
))
fig.add_vline(x=T_fixed, line_dash='dash', line_color='white', opacity=0.6,
               annotation_text='N=T (singular)', annotation_position='top right')
fig.update_layout(
    title='Curse of Dimensionality -- Condition Number vs N (T={} fixed)'.format(T_fixed),
    xaxis_title='Number of Assets (N)',
    yaxis_title='Condition Number (log scale)',
    yaxis_type='log',
    template='plotly_dark'
)
fig.show()
idx_eq = list(N_range).index(T_fixed) if T_fixed in N_range else np.searchsorted(N_range, T_fixed)
print('At N~T={}, condition number ~{:.1e}'.format(T_fixed, cond_num[min(idx_eq, len(cond_num)-1)]))

At N~T=252, condition number ~1.9e+04


---
## Part 4 | Dimension Reduction: PCA, RMT & Minimum Spanning Tree

### Principal Component Analysis

$$\Sigma = V \Lambda V^\top$$

- **PC1** (market factor) typically dominates variance.
- **Scree plot + Kaiser rule** (eigenvalue > 1 of the correlation matrix) identify meaningful components.
- **Cumulative variance** threshold (80-90%) determines the compression ratio needed.

In [36]:
pca       = SKPCA(n_components=N).fit(returns)
var_ratio = pca.explained_variance_ratio_
cum_var   = np.cumsum(var_ratio)
pc1_load  = pca.components_[0]

corr_eigvals = np.linalg.eigvalsh(corr.values)[::-1]
n_kaiser = (corr_eigvals > 1.0).sum()
n_80pct  = int(np.searchsorted(cum_var, 0.80)) + 1
n_90pct  = int(np.searchsorted(cum_var, 0.90)) + 1

print('PC1 variance share : {:.1f}%'.format(var_ratio[0]*100))
print('Kaiser rule        : {} components (corr eigenvalue > 1)'.format(n_kaiser))
print('80% threshold      : {} components'.format(n_80pct))
print('90% threshold      : {} components'.format(n_90pct))

fig = ps.make_subplots(rows=1, cols=3,
                        subplot_titles=['Scree Plot', 'Cumulative Variance', 'PC1 Loadings (Market Factor)'])
fig.add_trace(go.Bar(x=list(range(1,N+1)), y=var_ratio*100,
                      marker_color='rgba(100,149,237,0.7)', showlegend=False), row=1, col=1)
fig.add_hline(y=100/N, line_dash='dash', line_color='white', opacity=0.4,
               annotation_text='Random ({:.0f}%)'.format(100/N), row=1, col=1)

fig.add_trace(go.Scatter(x=list(range(1,N+1)), y=cum_var*100, mode='lines+markers',
                          line=dict(color='lightsalmon', width=2), showlegend=False), row=1, col=2)
for thr, lab in [(0.80,'80%'), (0.90,'90%')]:
    k = int(np.searchsorted(cum_var, thr)) + 1
    fig.add_vline(x=k, line_dash='dot', line_color='#00d4aa', row=1, col=2,
                   annotation_text='{} @ PC{}'.format(lab, k),
                   annotation_position='top right')

load_cols = [CLASS_COLORS[asset_class[a]] for a in ASSETS]
fig.add_trace(go.Bar(x=ASSETS, y=pc1_load, marker_color=load_cols, showlegend=False), row=1, col=3)

fig.update_xaxes(title_text='Component', row=1, col=1)
fig.update_yaxes(title_text='% Variance', row=1, col=1)
fig.update_xaxes(title_text='# Components', row=1, col=2)
fig.update_yaxes(title_text='Cumulative %', row=1, col=2)
fig.update_xaxes(tickangle=45, row=1, col=3)
fig.update_yaxes(title_text='Loading', row=1, col=3)
fig.update_layout(template='plotly_dark', height=430,
                   title='PCA -- Scree, Cumulative Variance, and PC1 (Market) Loadings by Asset Class')
fig.show()

PC1 variance share : 32.7%
Kaiser rule        : 4 components (corr eigenvalue > 1)
80% threshold      : 10 components
90% threshold      : 15 components


### Random Matrix Theory (RMT) — Filtering Estimation Noise

The **Marchenko-Pastur law** gives the eigenvalue range for a purely random $N \times T$ matrix:

$$\lambda_{\max}^{\text{noise}} = \sigma^2\!\left(1 + \sqrt{N/T}\right)^2$$

- Eigenvalues **below** $\lambda_{\max}$ are noise — replaced by their average (shrinkage).
- Eigenvalues **above** correspond to genuine systematic risk factors.
- The cleaned matrix has lower condition number and produces more stable MV weights.

In [37]:
q          = T / N
lam_max_mp = (1 + np.sqrt(1/q))**2   # sigma^2=1 (correlation matrix)
lam_min_mp = (1 - np.sqrt(1/q))**2
n_signal   = (corr_eigvals > lam_max_mp).sum()

print('Marchenko-Pastur  lambda_max = {:.3f}  lambda_min = {:.3f}'.format(lam_max_mp, lam_min_mp))
print('q = T/N = {:.1f}'.format(q))
print('Signal eigenvalues (> lambda_max): {}'.format(n_signal))

# Theoretical MP density
lam_rng = np.linspace(max(lam_min_mp, 1e-4), lam_max_mp, 300)
mp_pdf  = (q / (2*np.pi)) * np.sqrt((lam_max_mp - lam_rng)*(lam_rng - lam_min_mp)) / lam_rng

fig = go.Figure()
fig.add_trace(go.Histogram(x=corr_eigvals, nbinsx=25,
                             marker_color='rgba(100,149,237,0.6)',
                             name='Empirical eigenvalues', histnorm='probability density'))
fig.add_trace(go.Scatter(x=lam_rng, y=mp_pdf, mode='lines',
                          line=dict(color='lightsalmon', width=2, dash='dash'),
                          name='Marchenko-Pastur (noise)'))
fig.add_vline(x=lam_max_mp, line_dash='dot', line_color='#ff6b6b',
               annotation_text='lambda_max = {:.2f}'.format(lam_max_mp),
               annotation_position='top right')
sig_eigs = corr_eigvals[corr_eigvals > lam_max_mp]
fig.add_trace(go.Scatter(x=sig_eigs, y=np.full(len(sig_eigs), 0.05), mode='markers',
                          marker=dict(color='#00d4aa', size=10, symbol='triangle-up'),
                          name='Signal eigenvalues ({})'.format(len(sig_eigs))))
fig.update_layout(
    title='RMT: Empirical Eigenvalue Spectrum vs. Marchenko-Pastur Noise Distribution',
    xaxis_title='Eigenvalue', yaxis_title='Density',
    template='plotly_dark'
)
fig.show()

# ── Clean the correlation matrix ──────────────────────────────────────────────
ev, vecs = np.linalg.eigh(corr.values)
ev = ev[::-1]; vecs = vecs[:,::-1]
noise_mask  = ev < lam_max_mp
ev_clean    = np.where(noise_mask, ev[noise_mask].mean(), ev)
ev_clean   *= N / ev_clean.sum()           # rescale: trace = N
corr_clean  = vecs @ np.diag(ev_clean) @ vecs.T
np.fill_diagonal(corr_clean, 1.0)
cond_raw   = ev[0]  / max(ev[-1],  1e-12)
ev_c       = np.linalg.eigvalsh(corr_clean)
cond_clean = ev_c[-1] / max(ev_c[0], 1e-12)
print('\nCondition number -- raw: {:.0f}  cleaned: {:.0f}'.format(cond_raw, cond_clean))

Marchenko-Pastur  lambda_max = 1.352  lambda_min = 0.701
q = T/N = 37.8
Signal eigenvalues (> lambda_max): 4



Condition number -- raw: 22  cleaned: 16


---
## Final Exercise | Minimum Spanning Tree (MST)

**Steps** (from the presentation):

1. **Data preparation** — compute the log-return correlation matrix.
2. **Distance mapping** — $d_{ij} = \sqrt{2(1-\rho_{ij})}$.
3. **Tree construction** — Kruskal's algorithm (via `scipy.sparse.csgraph`).

**Interpretation:**
- **Hubs** (high-degree nodes) = systemic risk centres — cutting them fragments the portfolio.
- **Leaf nodes** (degree=1) = peripheral diversifiers.
- **Stress regime**: the tree compresses — hubs gain connections, average edge distance shrinks.

In [38]:
# ── Kruskal MST ──────────────────────────────────────────────────────────────
dist_full = np.sqrt(2*(1 - corr.values))
np.fill_diagonal(dist_full, 0)
mst_sparse = minimum_spanning_tree(csr_matrix(dist_full)).tocoo()
mst_edges  = list(zip(mst_sparse.row, mst_sparse.col, mst_sparse.data))
mst_edges += [(j,i,d) for i,j,d in mst_edges]  # symmetrize

degree = np.zeros(N, int)
for i,j,_ in mst_edges: degree[i] += 1
degree //= 2

# ── Spring layout (Fruchterman-Reingold) ──────────────────────────────────────
rng_lay = np.random.default_rng(7)
pos = rng_lay.uniform(-1, 1, (N, 2))
adj = np.zeros((N,N))
for i,j,d in mst_edges: adj[i,j] = 1/max(d, 1e-6)
for _ in range(300):
    delta = np.zeros_like(pos)
    for i in range(N):
        for j in range(N):
            if adj[i,j] > 0:
                diff = pos[i] - pos[j]
                dn   = max(np.linalg.norm(diff), 1e-6)
                delta[i] += adj[i,j] * (diff/dn) * (dn - 1.0/adj[i,j])
    pos -= 0.008 * delta
    pos -= pos.mean(0)

# ── Plotly network ────────────────────────────────────────────────────────────
fig = go.Figure()

# Edges (draw first so nodes appear on top)
for i,j,d in [(i,j,d) for i,j,d in mst_edges if i<j]:
    fig.add_trace(go.Scatter(
        x=[pos[i,0], pos[j,0], None],
        y=[pos[i,1], pos[j,1], None],
        mode='lines',
        line=dict(color='rgba(180,180,180,0.45)', width=max(1, int(3*(1-d/2)))),
        showlegend=False, hoverinfo='none'
    ))

# Nodes coloured by class, sized by degree
for cls, names in ASSET_CLASSES.items():
    idxs = [ASSETS.index(a) for a in names]
    fig.add_trace(go.Scatter(
        x=pos[idxs,0], y=pos[idxs,1],
        mode='markers+text',
        marker=dict(color=CLASS_COLORS[cls],
                     size=[9 + 5*degree[i] for i in idxs],
                     line=dict(color='white', width=1)),
        text=names, textposition='top center',
        textfont=dict(size=9, color='white'),
        name=cls
    ))

fig.update_layout(
    title='Minimum Spanning Tree -- Risk Skeleton of the 20-Asset Universe<br>'
          '<sup>Node size = MST degree.  Colour = asset class.  '
          'Edge thickness ~ correlation proximity.</sup>',
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    template='plotly_dark', height=600
)
fig.show()

top3 = np.argsort(degree)[::-1][:3]
print('Top 3 hubs:')
for i in top3:
    print('  {} [{}]  degree={}'.format(ASSETS[i], asset_class[ASSETS[i]], degree[i]))
leaves_mst = [ASSETS[i] for i in range(N) if degree[i]==1]
print('\nLeaf nodes (best diversifiers):', ', '.join(leaves_mst))

Top 3 hubs:
  BD03 [Bond]  degree=3
  EQ02 [Equity]  degree=2
  FX05 [FX]  degree=1

Leaf nodes (best diversifiers): EQ05, BD04, CM01, CM03, CM04, FX01, FX04, FX05


In [39]:
# ── Calm vs stress topology comparison ───────────────────────────────────────
def mst_degree(ret_subset):
    c = pd.DataFrame(ret_subset).corr().values
    d = np.sqrt(2*(1-c)); np.fill_diagonal(d, 0)
    m = minimum_spanning_tree(csr_matrix(d)).tocoo()
    deg = np.zeros(N, int)
    for i,j in zip(m.row, m.col): deg[i] += 1
    return deg, m.data.mean()

deg_calm,   avg_calm   = mst_degree(ret_stress.values[:T//3])
deg_stress, avg_stress = mst_degree(ret_stress.values[STRESS_START:STRESS_END])

fig = go.Figure()
fig.add_trace(go.Bar(x=ASSETS, y=deg_calm,   name='Calm',
                      marker_color='rgba(0,212,170,0.7)'))
fig.add_trace(go.Bar(x=ASSETS, y=deg_stress, name='Stress',
                      marker_color='rgba(255,107,107,0.7)'))
fig.update_layout(
    title='MST Node Degree: Calm vs. Stress -- Hubs Gain Connections During Stress',
    xaxis_title='Asset', yaxis_title='MST Degree',
    barmode='group', template='plotly_dark'
)
fig.show()
print('Avg MST edge distance -- calm: {:.3f}  stress: {:.3f}'.format(avg_calm, avg_stress))
print('(Shorter = higher correlations = less diversification)')

Avg MST edge distance -- calm: 0.978  stress: 0.433
(Shorter = higher correlations = less diversification)


---
## Conclusions

### Part 1 — Correlation & Diversification
The block-diagonal structure confirms within-class assets are far more correlated than across-class pairs. The rolling analysis shows average pairwise correlation spiking during the stress episode — diversification fails exactly when it is most needed.

### Part 2 — Tail Dependency & Partial Correlation
Within-class tail dependence is elevated, and the lower tail (crashes) is stronger than the upper (rallies) — classic asymmetric co-movement. Partial correlation after removing the market factor substantially reduces apparent linkages, isolating the direct pairwise relationships.

### Part 3 — Clustering & Curse of Dimensionality
HCA with Ward linkage on correlation distance recovers all four asset classes without any labels. K-Means confirms K=4 via the elbow. The condition number chart illustrates why covariance regularisation (shrinkage, RMT) is essential as N grows toward T.

### Part 4 — PCA, RMT & MST
- **PCA**: PC1 (market) dominates; 4-5 components explain ~80% of variance, matching the four latent factors.
- **RMT**: The Marchenko-Pastur boundary separates the small number of genuine signal eigenvalues from noise. Cleaning reduces the condition number, stabilising MV weights.
- **MST**: The risk skeleton reveals equity assets as central hubs while FX and commodity names sit at the periphery — the natural diversifiers. During stress the tree compresses: hubs attract more connections and average edge distance falls, reflecting the correlation convergence seen in Part 1.

> **Key takeaway**: Multi-asset portfolio construction requires understanding correlation at three levels — full matrix (MPT), cluster structure (HCA/K-Means), and network topology (MST). PCA + RMT tame the curse of dimensionality; the MST identifies which assets are true diversifiers vs systemic hubs.